# 02 — Data preparation

Preparation and validation of canonical inputs without modifying `data/raw/`. Reusable national
pipeline logic lives in `src/` and `scripts/`; this notebook preserves validated preparation evidence.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.paths import ensure_output_directories
ensure_output_directories()

print(f"Project root: {PROJECT_ROOT}")

Project root: /mnt/data/wildfire_property_screening_portugal


In [2]:
from src.config import SPATIAL
from src.paths import RAW_DATA_DIR, INTERIM_DATA_DIR, PROCESSED_DATA_DIR

print(f"Analysis CRS: {SPATIAL.analysis_crs}")
print(f"Grid size: {SPATIAL.grid_size_metres} m")
print(f"Context buffer: {SPATIAL.context_buffer_metres} m")
print("Preparation logic will be added after the feasibility pilot confirms source schemas.")

Analysis CRS: EPSG:3763
Grid size: 1000 m
Context buffer: 2000 m
Preparation logic will be added after the feasibility pilot confirms source schemas.


## CAOP 2025 reference-layer preparation

Read the CAOP 2025 GeoPackage from the raw ZIP without changing it. The archive is extracted only to a system temporary directory while this cell runs. The validated mainland boundary and municipality reference layers are written to `data/processed/reference/`.

In [ ]:
from tempfile import TemporaryDirectory
from zipfile import ZipFile

import geopandas as gpd
from pyproj import CRS

CAOP_ARCHIVE = RAW_DATA_DIR / "boundaries" / "dgt_caop" / "CAOP_Continente_2025-gpkg.zip"
REFERENCE_DIR = PROCESSED_DATA_DIR / "reference"
BOUNDARY_OUTPUT = REFERENCE_DIR / "mainland_boundary_caop2025.gpkg"
MUNICIPALITIES_OUTPUT = REFERENCE_DIR / "municipalities_caop2025.gpkg"
EXPECTED_CRS = CRS.from_epsg(3763)

if not CAOP_ARCHIVE.is_file():
    raise FileNotFoundError(f"Missing CAOP archive: {CAOP_ARCHIVE}")


def validate_layer(name: str, frame: gpd.GeoDataFrame, expected_count: int) -> None:
    if frame.crs is None or CRS.from_user_input(frame.crs) != EXPECTED_CRS:
        raise ValueError(f"{name}: expected EPSG:3763, found {frame.crs}")
    if len(frame) != expected_count:
        raise ValueError(f"{name}: expected {expected_count} features, found {len(frame)}")
    if frame.geometry.isna().any() or frame.geometry.is_empty.any():
        raise ValueError(f"{name}: geometry contains null or empty values")
    if not frame.geometry.is_valid.all():
        raise ValueError(f"{name}: geometry contains invalid values")


with TemporaryDirectory(prefix="caop2025_") as temporary_directory:
    temporary_directory = Path(temporary_directory)
    with ZipFile(CAOP_ARCHIVE) as archive:
        corrupt_member = archive.testzip()
        if corrupt_member is not None:
            raise ValueError(f"CAOP archive failed CRC validation: {corrupt_member}")

        gpkg_members = [
            member for member in archive.namelist()
            if member.lower().endswith(".gpkg") and not member.endswith("/")
        ]
        if len(gpkg_members) != 1:
            raise ValueError(f"Expected one GeoPackage in the CAOP archive, found {gpkg_members}")
        extracted_gpkg = Path(archive.extract(gpkg_members[0], path=temporary_directory))

    mainland_source = gpd.read_file(extracted_gpkg, layer="cont_nuts1")
    municipalities_source = gpd.read_file(extracted_gpkg, layer="cont_municipios")

    validate_layer("cont_nuts1", mainland_source, expected_count=1)
    if mainland_source["nuts1"].tolist() != ["Continente"]:
        raise ValueError("cont_nuts1 must contain exactly one feature where nuts1 == 'Continente'")

    validate_layer("cont_municipios", municipalities_source, expected_count=278)
    if municipalities_source["dtmn"].isna().any() or not municipalities_source["dtmn"].is_unique:
        raise ValueError("cont_municipios.dtmn must be non-null and unique")

    mainland_boundary = mainland_source.loc[:, ["nuts1", "geometry"]].copy()
    municipalities = municipalities_source.loc[:, ["dtmn", "municipio", "geometry"]].copy()

    REFERENCE_DIR.mkdir(parents=True, exist_ok=True)
    for output_path in (BOUNDARY_OUTPUT, MUNICIPALITIES_OUTPUT):
        if output_path.exists():
            output_path.unlink()

    mainland_boundary.to_file(BOUNDARY_OUTPUT, layer="mainland_boundary_caop2025", driver="GPKG")
    municipalities.to_file(MUNICIPALITIES_OUTPUT, layer="municipalities_caop2025", driver="GPKG")

print(f"Validated cont_nuts1: {len(mainland_boundary)} feature; CRS {mainland_boundary.crs}")
print(f"Validated cont_municipios: {len(municipalities)} features; unique non-null dtmn; CRS {municipalities.crs}")
print(f"Wrote boundary: {BOUNDARY_OUTPUT}")
print(f"Wrote municipalities: {MUNICIPALITIES_OUTPUT}")

## ICNF 2024 burned-area pilot preparation

Read the ICNF 2024 raw archive without changing it, extract its Shapefile only to a system temporary directory, and clip the geometry to the processed CAOP mainland boundary. `AreaHaSIG` is retained unchanged as the official reported-area reference field; it is not used to calculate geometry. This section does not create the 1 km grid or any target.

In [ ]:
from tempfile import TemporaryDirectory
from zipfile import ZipFile

import geopandas as gpd
from pyproj import CRS

ICNF_ARCHIVE = RAW_DATA_DIR / "wildfire" / "icnf_burned_areas" / "ardida_2024.zip"
MAINLAND_BOUNDARY = PROCESSED_DATA_DIR / "reference" / "mainland_boundary_caop2025.gpkg"
ICNF_OUTPUT_DIR = PROCESSED_DATA_DIR / "labels" / "icnf_burned_areas"
ICNF_OUTPUT = ICNF_OUTPUT_DIR / "icnf_burned_areas_2024.gpkg"
EXPECTED_CRS = CRS.from_epsg(3763)
REQUIRED_FIELDS = ["Cod_SGIF", "Ano", "DH_Inicio", "AreaHaSIG"]
REQUIRED_MEMBERS = [
    "ardida_2024.shp",
    "ardida_2024.shx",
    "ardida_2024.dbf",
    "ardida_2024.prj",
]

if not ICNF_ARCHIVE.is_file():
    raise FileNotFoundError(f"Missing ICNF archive: {ICNF_ARCHIVE}")
if not MAINLAND_BOUNDARY.is_file():
    raise FileNotFoundError(f"Missing processed mainland boundary: {MAINLAND_BOUNDARY}")

mainland_boundary = gpd.read_file(MAINLAND_BOUNDARY, layer="mainland_boundary_caop2025")
if mainland_boundary.crs is None or CRS.from_user_input(mainland_boundary.crs) != EXPECTED_CRS:
    raise ValueError(f"Mainland boundary: expected EPSG:3763, found {mainland_boundary.crs}")
if len(mainland_boundary) != 1 or mainland_boundary.geometry.isna().any() or mainland_boundary.geometry.is_empty.any():
    raise ValueError("Mainland boundary must contain one non-empty geometry")
if not mainland_boundary.geometry.is_valid.all():
    raise ValueError("Mainland boundary geometry is invalid")

with TemporaryDirectory(prefix="icnf_burned_areas_2024_") as temporary_directory:
    temporary_directory = Path(temporary_directory)
    with ZipFile(ICNF_ARCHIVE) as archive:
        corrupt_member = archive.testzip()
        if corrupt_member is not None:
            raise ValueError(f"ICNF archive failed CRC validation: {corrupt_member}")
        archive_members = archive.namelist()
        missing_members = sorted(set(REQUIRED_MEMBERS) - set(archive_members))
        if missing_members:
            raise ValueError(f"Missing Shapefile components: {missing_members}")
        for member in REQUIRED_MEMBERS:
            archive.extract(member, path=temporary_directory)

    burned_areas = gpd.read_file(temporary_directory / "ardida_2024.shp")

    if burned_areas.crs is None or CRS.from_user_input(burned_areas.crs) != EXPECTED_CRS:
        raise ValueError(f"ICNF 2024: expected EPSG:3763, found {burned_areas.crs}")
    if len(burned_areas) != 1558:
        raise ValueError(f"ICNF 2024: expected 1,558 features, found {len(burned_areas)}")
    if burned_areas.geometry.isna().any() or burned_areas.geometry.is_empty.any():
        raise ValueError("ICNF 2024: geometry contains null or empty values before clipping")
    if not burned_areas.geometry.is_valid.all():
        raise ValueError("ICNF 2024: geometry contains invalid values before clipping")
    missing_fields = [field for field in REQUIRED_FIELDS if field not in burned_areas.columns]
    if missing_fields:
        raise ValueError(f"ICNF 2024: missing required fields {missing_fields}")
    if not burned_areas["Ano"].eq(2024).all():
        raise ValueError("ICNF 2024: all Ano values must equal 2024")

    mainland_geometry = mainland_boundary.geometry.union_all()
    if not burned_areas.geometry.intersects(mainland_geometry).all():
        raise ValueError("ICNF 2024: every feature must intersect the mainland boundary")

    burned_areas_clipped = gpd.clip(burned_areas, mainland_boundary)

burned_areas_clipped = burned_areas_clipped.loc[:, ["Cod_SGIF", "Ano", "DH_Inicio", "AreaHaSIG", "geometry"]].copy()
if burned_areas_clipped.geometry.isna().any() or burned_areas_clipped.geometry.is_empty.any():
    raise ValueError("Clipped ICNF 2024 geometry contains null or empty values")
if not burned_areas_clipped.geometry.is_valid.all():
    raise ValueError("Clipped ICNF 2024 geometry contains invalid values")

ICNF_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if ICNF_OUTPUT.exists():
    ICNF_OUTPUT.unlink()
burned_areas_clipped.to_file(ICNF_OUTPUT, layer="icnf_burned_areas_2024", driver="GPKG")

icnf_output = gpd.read_file(ICNF_OUTPUT, layer="icnf_burned_areas_2024")
if icnf_output.crs is None or CRS.from_user_input(icnf_output.crs) != EXPECTED_CRS:
    raise ValueError(f"ICNF output: expected EPSG:3763, found {icnf_output.crs}")
if list(icnf_output.columns) != ["Cod_SGIF", "Ano", "DH_Inicio", "AreaHaSIG", "geometry"]:
    raise ValueError(f"ICNF output: unexpected fields {list(icnf_output.columns)}")
if icnf_output.geometry.isna().any() or icnf_output.geometry.is_empty.any():
    raise ValueError("ICNF output: geometry contains null or empty values")
if not icnf_output.geometry.is_valid.all():
    raise ValueError("ICNF output: geometry contains invalid values")
if not icnf_output["Ano"].eq(2024).all():
    raise ValueError("ICNF output: all Ano values must equal 2024")

print(f"Validated ICNF source: {len(burned_areas)} non-empty valid geometries; CRS {burned_areas.crs}")
print(f"Clipped ICNF output: {len(icnf_output)} non-empty valid geometries; CRS {icnf_output.crs}")
print(f"Wrote ICNF 2024 labels: {ICNF_OUTPUT}")

## CLC 2018 mainland vector inspection

Validate the existing mainland CLC 2018 interim derivative. This section reads the immutable raw-download provenance and the interim GeoPackage; it does not create a grid, calculate land-cover features, or modify either input.

In [ ]:
from src.clc_validation import validate_clc_2018_mainland

clc_2018_mainland_validation = validate_clc_2018_mainland(PROJECT_ROOT)
print(clc_2018_mainland_validation)